# P9 - Mission 1 : Analysez le catalogue

Analyse du catalogue MeloWave pour Camille Roussel (Directrice des études musicales).


## 1. Imports

In [1]:
# Installation des dependances (a executer une seule fois)
!pip install statsmodels scikit-posthocs


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from scipy import stats

# Embarque la librairie Plotly dans les sorties du notebook : sans cette ligne,
# les graphiques n'apparaissent pas dans l'export HTML (livrable attendu).
pio.renderers.default = 'notebook'


## 2. Chargement des donnees

In [3]:
df = pd.read_csv("../data/spotify-tracks-dataset-detailed.csv")
print(df.shape)
df.head()


(114000, 20)


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


## 3. Audit qualite des donnees

### 3.1 Premier coup d'oeil : types et valeurs manquantes

In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114000 entries, 0 to 113999
Data columns (total 20 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   track_id          114000 non-null  object 
 1   artists           113999 non-null  object 
 2   album_name        113999 non-null  object 
 3   track_name        113999 non-null  object 
 4   popularity        114000 non-null  int64  
 5   duration_ms       114000 non-null  int64  
 6   explicit          114000 non-null  bool   
 7   danceability      114000 non-null  float64
 8   energy            114000 non-null  float64
 9   key               114000 non-null  int64  
 10  loudness          114000 non-null  float64
 11  mode              114000 non-null  int64  
 12  speechiness       114000 non-null  float64
 13  acousticness      114000 non-null  float64
 14  instrumentalness  114000 non-null  float64
 15  liveness          114000 non-null  float64
 16  valence           11

In [5]:
df.isnull().sum()


track_id            0
artists             1
album_name          1
track_name          1
popularity          0
duration_ms         0
explicit            0
danceability        0
energy              0
key                 0
loudness            0
mode                0
speechiness         0
acousticness        0
instrumentalness    0
liveness            0
valence             0
tempo               0
time_signature      0
track_genre         0
dtype: int64

In [6]:
df.describe()


,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature
count,114000.000000,1.140000e+05,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000,114000.000000
mean,33.238535,2.280292e+05,0.566800,0.641383,5.309140,-8.258960,0.637553,0.084652,0.314910,0.156050,0.213553,0.474068,122.147837,3.904035
std,22.305078,1.072977e+05,0.173542,0.251529,3.559987,5.029337,0.480709,0.105732,0.332523,0.309555,0.190378,0.259261,29.978197,0.432621
min,0.000000,0.000000e+00,0.000000,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,17.000000,1.740660e+05,0.456000,0.472000,2.000000,-10.013000,0.000000,0.035900,0.016900,0.000000,0.098000,0.260000,99.218750,4.000000
50%,35.000000,2.129060e+05,0.580000,0.685000,5.000000,-7.004000,1.000000,0.048900,0.169000,0.000042,0.132000,0.464000,122.017000,4.000000
75%,50.000000,2.615060e+05,0.695000,0.854000,8.000000,-5.003000,1.000000,0.084500,0.598000,0.049000,0.273000,0.683000,140.071000,4.000000
max,100.000000,5.237295e+06,0.985000,1.000000,11.000000,4.532000,1.000000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5.000000


In [7]:
# Ligne avec artists/album_name/track_name manquants
df[df['artists'].isnull()]


,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
65900,1kR4gIb7nGxHPI3D2ifs59,NaN,NaN,NaN,0,0,False,0.501,0.583,7,-9.46,0,0.0605,0.69,0.00396,0.0747,0.734,138.391,4,k-pop


In [8]:
# Doublons stricts (toutes colonnes identiques) et doublons de track_id (multi-genres)
print("Doublons stricts :", df.duplicated().sum())
print("Doublons track_id :", df['track_id'].duplicated().sum())


Doublons stricts : 450
Doublons track_id : 24259


### 3.2 Nettoyage

**Décisions prises** (justifiées dans `../CAHIER_DES_CHARGES.md`) :
- La ligne avec `artists`/`album_name`/`track_name` manquants a aussi `duration_ms = 0` (impossible pour un vrai morceau) → ligne corrompue, supprimée.
- Les 450 doublons stricts sont supprimés (pure redondance).
- Les 24 259 doublons de `track_id` sont des réinscriptions légitimes d'un même morceau sous plusieurs genres (vérifié sur exemple). Le traitement dépend de l'analyse : deux dataframes sont donc construits.


In [9]:
# Suppression de la ligne corrompue (duration_ms == 0) et des doublons stricts
df_clean = df[df['duration_ms'] != 0].copy()
df_clean = df_clean.drop_duplicates()

# Conversion de la duree en secondes pour faciliter la lecture des graphiques
df_clean['duration_sec'] = df_clean['duration_ms'] / 1000

print("Lignes avant :", df.shape[0])
print("Lignes apres suppression ligne corrompue + doublons stricts :", df_clean.shape[0])


Lignes avant : 114000
Lignes apres suppression ligne corrompue + doublons stricts : 113549


In [10]:
# df_tracks : une ligne par morceau (dedupliquee sur track_id)
# -> a utiliser pour tout ce qui ne porte pas sur le genre : classements, description du catalogue, Q1 (correlation audio)
df_tracks = df_clean.drop_duplicates(subset='track_id', keep='first').copy()

# df_genre : une ligne par (morceau, genre), doublons track_id conserves
# -> a utiliser quand le genre est la variable etudiee : Q2 (popularite/genre), Q3 (explicite/genre)
df_genre = df_clean.copy()

print("df_tracks :", df_tracks.shape[0], "morceaux uniques")
print("df_genre  :", df_genre.shape[0], "lignes (morceau x genre)")


df_tracks : 89740 morceaux uniques
df_genre  : 113549 lignes (morceau x genre)


### 3.3 Quel dataframe pour quelle analyse ? (arbitrage documente)

Le dataset n'est pas un catalogue naturel : il contient **exactement 1000 morceaux par genre, pour 114 genres**, soit un echantillon **stratifie et parfaitement equilibre**. Les doublons de `track_id` viennent de la : un morceau rattache a plusieurs genres est tire dans plusieurs strates.

Deux lectures sont donc possibles, et chacune correspond a une question differente :

| Lecture | Dataframe | Ce qu'on mesure |
|---|---|---|
| Le dataset tel que fourni, equilibre par genre | `df_genre` (113 549 lignes) | les caracteristiques du catalogue par strate de genre |
| Les morceaux reellement distincts | `df_tracks` (89 740 lignes) | les caracteristiques des morceaux, chacun compte une fois |

**Regle retenue dans ce notebook** :
- `df_genre` pour la description du dataset et toute analyse ou le genre intervient (sections 4 et 5). C'est le choix de reference, qui preserve l'equilibre a 1000 morceaux par genre, indispensable pour comparer les genres entre eux a poids egal.
- `df_tracks` uniquement la ou compter plusieurs fois le meme titre serait factuellement faux : classements d'artistes (section 4.6), investigation de morceaux precis, et analyses portant sur le morceau lui-meme (section 6).

La cellule suivante quantifie l'ecart entre les deux lectures, pour verifier que le choix n'a pas d'incidence majeure sur les conclusions.

In [11]:
# Comparaison chiffree des deux lectures, sur les indicateurs les plus sensibles
from scipy.stats import pearsonr

comparaison = pd.DataFrame({
    'df_genre (dataset stratifie)': [
        len(df_genre),
        df_genre['popularity'].mean(),
        (df_genre['popularity'] == 0).mean() * 100,
        df_genre['duration_sec'].mean(),
        df_genre['energy'].mean(),
        pearsonr(df_genre['energy'], df_genre['loudness'])[0],
    ],
    'df_tracks (morceaux distincts)': [
        len(df_tracks),
        df_tracks['popularity'].mean(),
        (df_tracks['popularity'] == 0).mean() * 100,
        df_tracks['duration_sec'].mean(),
        df_tracks['energy'].mean(),
        pearsonr(df_tracks['energy'], df_tracks['loudness'])[0],
    ],
}, index=['nombre de lignes', 'popularite moyenne', 'part de popularite a 0 (%)',
          'duree moyenne (s)', 'energy moyenne', 'correlation energy/loudness (r)'])

comparaison['ecart relatif (%)'] = ((comparaison.iloc[:, 1] - comparaison.iloc[:, 0])
                                     / comparaison.iloc[:, 0] * 100)
comparaison.round(3)


,df_genre (dataset stratifie),df_tracks (morceaux distincts),ecart relatif (%)
nombre de lignes,113549.000,89740.000,-20.968
popularite moyenne,33.324,33.199,-0.377
part de popularite a 0 (%),13.953,10.527,-24.551
duree moyenne (s),228.081,229.144,0.466
energy moyenne,0.642,0.634,-1.189
correlation energy/loudness (r),0.761,0.759,-0.243


**Conclusion de l'arbitrage** : sur la plupart des indicateurs, les deux lectures donnent des resultats tres proches. La correlation energy/loudness passe de 0,761 a 0,759 selon le dataframe, l'energy moyenne et la duree moyenne varient de moins de 1%. Le choix du dataframe ne change donc pas les conclusions metier.

**Une exception notable** : la part de morceaux a popularite nulle vaut **14,0% dans `df_genre` contre 10,5% dans `df_tracks`**. L'ecart n'est pas un artefact : il signifie que les morceaux a popularite nulle sont rattaches a davantage de genres que la moyenne (environ 1,7 strate chacun, contre 1,27 pour un morceau quelconque). Autrement dit, les titres confidentiels sont plus souvent classes dans plusieurs genres de niche a la fois.

C'est precisement le genre d'ecart qu'il faut connaitre avant d'annoncer un chiffre a Camille : selon la lecture retenue, on lui dit qu'un morceau sur dix ou un sur sept n'a jamais ete ecoute recemment.

## 4. Description du catalogue

Conformement a la regle posee en section 3.3 :
- **`df_genre`** pour les statistiques descriptives, les distributions et les comparaisons entre genres (sections 4.1 a 4.5, 4.7, 4.8)
- **`df_tracks`** pour les classements d'artistes et de morceaux (section 4.6) et pour les investigations portant sur des titres identifies, ou compter plusieurs fois le meme morceau serait factuellement faux

### 4.1 Statistiques descriptives

In [12]:
# Variables audio continues : une moyenne a du sens sur chacune.
# key, mode et time_signature en sont exclues volontairement : ce sont des etiquettes
# (hauteur de note, mineur/majeur, mesure), pas des quantites. Elles sont traitees en 6.3.
audio_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_sec']

df_genre[audio_cols + ['popularity']].describe()


,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_sec,popularity
count,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000,113549.000000
mean,0.567031,0.642091,-8.243408,0.084674,0.314064,0.155703,0.213613,0.474205,122.175745,228.081371,33.324433
std,0.173409,0.251053,5.011422,0.105762,0.331906,0.309217,0.190462,0.259204,29.972954,106.413099,22.283855
min,0.000000,0.000000,-49.531000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,8.586000,0.000000
25%,0.456000,0.473000,-9.998000,0.035900,0.016800,0.000000,0.098000,0.260000,99.296000,174.184000,17.000000
50%,0.580000,0.685000,-6.997000,0.048900,0.168000,0.000041,0.132000,0.464000,122.020000,213.000000,35.000000
75%,0.695000,0.854000,-5.001000,0.084500,0.596000,0.048700,0.273000,0.683000,140.074000,261.588000,50.000000
max,0.985000,1.000000,4.532000,0.965000,0.996000,1.000000,1.000000,0.995000,243.372000,5237.295000,100.000000


### 4.2 Les genres du catalogue

Vue d'ensemble des genres disponibles et de leur volume respectif.

In [13]:
print(f"Nombre de genres distincts : {df_tracks['track_genre'].nunique()}")
print()
print(sorted(df_tracks['track_genre'].unique()))


Nombre de genres distincts : 113

['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient', 'anime', 'black-metal', 'bluegrass', 'blues', 'brazil', 'breakbeat', 'british', 'cantopop', 'chicago-house', 'children', 'chill', 'classical', 'club', 'comedy', 'country', 'dance', 'dancehall', 'death-metal', 'deep-house', 'detroit-techno', 'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm', 'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk', 'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove', 'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle', 'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian', 'indie', 'indie-pop', 'industrial', 'iranian', 'j-dance', 'j-idol', 'j-pop', 'j-rock', 'jazz', 'k-pop', 'kids', 'latin', 'latino', 'malay', 'mandopop', 'metal', 'metalcore', 'minimal-techno', 'mpb', 'new-age', 'opera', 'pagode', 'party', 'piano', 'pop', 'pop-film', 'power-pop', 'progressive-house', 'psych-rock', 'punk', 'punk-rock', 'r-n-

### 4.3 Distribution de la popularite

In [14]:
fig = px.histogram(df_genre, x='popularity', nbins=50, title='Distribution de la popularite (df_genre)')
fig.show()

print("Part de morceaux a popularite = 0 :", (df_genre['popularity'] == 0).mean().round(3))


Part de morceaux a popularite = 0 : 0.14


### 4.4 Cas particuliers : valeurs a zero sur tempo et danceability

Le `describe()` de la section 4.1 montre un minimum de 0 sur `tempo`, ce qui est physiquement impossible pour un morceau reel. Verification ci-dessous.

In [15]:
print(f"Morceaux a tempo = 0 : {(df_tracks['tempo'] == 0).sum()}")


Morceaux a tempo = 0 : 157


In [16]:
# Profil des morceaux a tempo = 0 sur les autres variables
df_tracks[df_tracks['tempo'] == 0][['danceability', 'energy', 'loudness', 'popularity', 'duration_sec']].describe()


,danceability,energy,loudness,popularity,duration_sec
count,157.0,157.000000,157.000000,157.000000,157.000000
mean,0.0,0.122914,-22.092503,37.197452,171.830713
std,0.0,0.250220,9.677911,11.083853,403.350691
min,0.0,0.000000,-49.531000,0.000000,8.586000
25%,0.0,0.000020,-27.035000,33.000000,74.700000
50%,0.0,0.000638,-20.356000,35.000000,93.451000
75%,0.0,0.062800,-16.680000,41.000000,147.000000
max,0.0,0.999000,-0.689000,85.000000,3601.693000


In [17]:
# Repartition par genre de ces morceaux
genres_tempo_nul = df_tracks[df_tracks['tempo'] == 0]['track_genre'].value_counts()

fig = px.bar(x=genres_tempo_nul.values, y=genres_tempo_nul.index, orientation='h', text=genres_tempo_nul.values,
             labels={'x': 'nombre de morceaux', 'y': ''},
             title=f'Morceaux a tempo nul par genre ({genres_tempo_nul.sum()} morceaux)')
fig.update_yaxes(autorange='reversed')   # genre le plus frequent en haut
fig.show()

genres_tempo_nul


track_genre
sleep          138
guitar           4
iranian          4
ambient          3
world-music      3
opera            2
jazz             1
romance          1
show-tunes       1
Name: count, dtype: int64

**Conclusion** : 157 morceaux ont `tempo = 0`, et tous ont aussi `danceability = 0` (moyenne et ecart-type nuls). En revanche `energy`, `loudness`, `popularity` et `duration_sec` sont parfaitement valides et varies chez eux.

La repartition par genre est sans ambiguite : **138 des 157 (88%) sont du genre 'sleep'**, le reste se repartit sur guitar, iranian, ambient, world-music, opera, jazz, romance, show-tunes. Ce sont des genres ou un tempo dansant n'a pas de sens musical.

**Decision** : contrairement a la ligne corrompue supprimee en section 3.2 (`duration_ms = 0` avec toutes les metadonnees absentes), il s'agit ici de **valeurs reelles et legitimes** produites par l'algorithme audio de Spotify pour de la musique sans rythme detectable. **Lignes conservees telles quelles**, sans suppression ni imputation.

### 4.5 Asymetrie (skewness) et aplatissement (kurtosis)

In [18]:
from scipy.stats import skew, kurtosis

cols_to_check = audio_cols + ['popularity']

shape_stats = pd.DataFrame({
    'skewness': [skew(df_genre[c]) for c in cols_to_check],
    'kurtosis': [kurtosis(df_genre[c]) for c in cols_to_check]
}, index=cols_to_check)

shape_stats.round(2)


,skewness,kurtosis
danceability,-0.40,-0.18
energy,-0.60,-0.52
loudness,-2.01,5.96
speechiness,4.64,28.79
acousticness,0.73,-0.94
instrumentalness,1.74,1.28
liveness,2.11,4.38
valence,0.11,-1.03
tempo,0.23,-0.11
duration_sec,10.81,338.58


**Lecture rapide** :
- skewness proche de 0 = distribution symetrique ; positif = longue queue a droite ; negatif = longue queue a gauche
- kurtosis (excess) proche de 0 = comme une loi normale ; positif = queues epaisses (valeurs extremes frequentes) ; negatif = distribution plus plate

**Interpretation par variable** (calcul sur `df_genre`) :

| Variable | Skewness | Kurtosis | Lecture |
|---|---|---|---|
| danceability | -0,40 | -0,18 | quasi symetrique, legere concentration vers les valeurs hautes |
| energy | -0,60 | -0,52 | penche vers les valeurs hautes, distribution plutot plate (catalogue majoritairement energique) |
| loudness | -2,01 | 5,96 | forte asymetrie a gauche et queues epaisses : les morceaux tres silencieux des genres calmes tirent la distribution (cf. 4.8) |
| speechiness | 4,64 | 28,79 | **la plus asymetrique apres la duree** : l'immense majorite des morceaux contient peu de parole, quelques-uns en sont satures |
| acousticness | 0,73 | -0,94 | penche vers les valeurs basses (catalogue majoritairement non acoustique), tres etalee |
| instrumentalness | 1,74 | 1,28 | la plupart des morceaux sont chantes, une minorite est purement instrumentale |
| liveness | 2,11 | 4,38 | la plupart des enregistrements sont en studio, quelques captations live ressortent nettement |
| valence | 0,11 | -1,03 | symetrique mais tres plate : la positivite musicale se repartit uniformement |
| tempo | 0,23 | -0,11 | la plus proche d'une distribution normale de toutes les variables audio |
| duration_sec | 10,81 | 338,58 | valeurs extremes : voir l'investigation ci-dessous |
| popularity | 0,04 | -0,92 | symetrique en apparence, mais l'histogramme de la section 4.3 revele un pic isole a 0 que ces deux indicateurs ne captent pas |

**Point de methode** : la popularite illustre la limite de ces indicateurs. Une skewness de 0,04 suggere une distribution symetrique, alors que 14% des lignes sont concentrees sur la seule valeur 0. Skewness et kurtosis ne remplacent pas l'examen visuel de la distribution.

In [19]:
# Histogrammes de toutes les variables audio + popularite, pour visualiser la forme des distributions
for col in cols_to_check:
    fig = px.histogram(df_genre, x=col, nbins=50, title=f'Distribution de {col}')
    fig.show()


In [20]:
df_tracks['duration_sec'].sort_values(ascending=False).head(10)


73617     5237.295
10935     4789.026
24348     4730.302
73840     4563.897
13344     4447.520
13245     4339.826
13195     4334.721
27926     4246.206
101390    4120.258
45063     3876.276
Name: duration_sec, dtype: float64

In [21]:
fig = px.box(df_tracks, x='duration_sec', title='Boxplot duree (secondes)')
fig.show()

In [22]:
df_tracks.sort_values('duration_sec', ascending=False)[['track_name', 'track_genre', 'duration_sec']].head(10)

,track_name,track_genre,duration_sec
73617,Unity (Voyage Mix) Pt. 1,minimal-techno,5237.295
10935,Crossing Wires 002 - Continuous DJ Mix,breakbeat,4789.026
24348,The Lab 03 - Continuous DJ Mix Part 1,detroit-techno,4730.302
73840,Amnesia Ibiza Underground 10 DJ Mix,minimal-techno,4563.897
13344,House of Om - Mark Farina - Continuous Mix,chicago-house,4447.520
13245,Live In Tokyo - Continuous Mix,chicago-house,4339.826
13195,Greenhouse Construction,chicago-house,4334.721
27926,"NQ State of Mind, Vol. 1 - Continuous DJ Mix",drum-and-bass,4246.206
101390,Ocean Waves Sounds,sleep,4120.258
45063,Internal Flight (Remastered),guitar,3876.276


In [23]:
Q1 = df_tracks['duration_sec'].quantile(0.25)
Q3 = df_tracks['duration_sec'].quantile(0.75)
IQR = Q3 - Q1
seuil_haut = Q3 + 1.5 * IQR
outliers_duration = df_tracks[df_tracks['duration_sec'] > seuil_haut]
print(f"{len(outliers_duration)} morceaux au-dessus du seuil IQR ({seuil_haut:.0f}s)")

4124 morceaux au-dessus du seuil IQR (401s)


In [24]:
outliers_duration['duration_sec'].describe()

count    4124.000000
mean      528.673833
std       300.410207
min       401.186000
25%       426.504500
50%       461.806500
75%       530.542000
max      5237.295000
Name: duration_sec, dtype: float64

**Constat sur `duration_sec`** : skewness = 10,81 et kurtosis = 338,58 (calcul sur `df_genre`), extremement eleves. Verification :

*L'investigation des morceaux extremes ci-dessus est menee sur `df_tracks` : elle liste des titres precis, et sur `df_genre` un meme mix rattache a plusieurs genres apparaitrait plusieurs fois dans le classement.*

- Les 10 valeurs les plus hautes (jusqu'a 5237 s, soit 87 minutes) sont des mix DJ continus (`track_name` contenant 'Continuous Mix' ou 'DJ Mix'), sur des genres electro (minimal-techno, breakbeat, detroit-techno, chicago-house, drum-and-bass), auxquels s'ajoutent un morceau ambient du genre 'sleep' et un instrumental guitare. Donnees reelles et legitimes, pas une erreur.
- Methode IQR : seuil haut a 401 s, **4124 morceaux** detectes comme outliers (4,6% des morceaux distincts). Mais la mediane de ce groupe est a 462 s, soit 7,7 minutes : ce sont des morceaux **normaux**, juste un peu longs.

**Conclusion** : sur une distribution aussi asymetrique, la methode IQR sur-detecte massivement des faux positifs. Seule la queue extreme (au-dela d'environ 2000 s) correspond a de vraies singularites, et ce sont des mix, pas des erreurs. Traitement retenu : **aucune suppression**, les donnees sont conservees telles quelles et la duree n'est pas filtree pour les tests statistiques.

### 4.6 Classements artistes et morceaux

*Analyse menee sur `df_tracks` (dedupliquee) : compter plusieurs fois le meme titre gonflerait artificiellement le nombre de morceaux d'un artiste. Un comptage sur `df_genre` est fait en fin de section, uniquement pour mesurer cet ecart.*

In [25]:
# Eclatement des featurings (separateur ';') pour compter chaque artiste individuellement
df_artists = df_tracks.copy()
df_artists['artists'] = df_artists['artists'].str.split(';')
df_artists = df_artists.explode('artists')
df_artists['artists'] = df_artists['artists'].str.strip()

print("Lignes avant eclatement :", df_tracks.shape[0])
print("Lignes apres eclatement :", df_artists.shape[0])


Lignes avant eclatement : 89740
Lignes apres eclatement : 123424


**Classement 1 : artistes les plus presents (nombre de morceaux)**

In [26]:
top_presence = df_artists['artists'].value_counts().head(15)

fig = px.bar(x=top_presence.values, y=top_presence.index, orientation='h', text=top_presence.values,
             labels={'x': 'nombre de morceaux', 'y': ''},
             title='Classement 1 : les 15 artistes les plus presents du catalogue')
fig.update_yaxes(autorange='reversed')   # artiste le plus present en haut
fig.show()

top_presence


artists
George Jones               332
Pritam                     323
Wolfgang Amadeus Mozart    305
Arijit Singh               259
Hank Williams              243
my little airport          171
Glee Cast                  170
Yuvan Shankar Raja         167
Hillsong Worship           167
BTS                        165
J Balvin                   156
Linkin Park                154
Hans Zimmer                153
The Beatles                150
Scooter                    143
Name: count, dtype: int64

In [27]:
df_artists['artists'].value_counts().describe()

count    29858.000000
mean         4.133700
std          9.298884
min          1.000000
25%          1.000000
50%          1.000000
75%          4.000000
max        332.000000
Name: count, dtype: float64

In [28]:
(df_artists['artists'].value_counts() >= 8).sum()

np.int64(3593)

**Justification du seuil de 8 morceaux minimum** :
- Distribution du nombre de morceaux par artiste (29 858 artistes uniques apres eclatement des featurings) : mediane = 1, 25e percentile = 1, 75e percentile = 4, max = 332
- La moitie des artistes n'a qu'1 seul morceau : un hit isole fausserait leur popularite moyenne
- Un seuil a 8 exclut ces cas non representatifs, tout en conservant 3593 artistes (12% du total) suffisamment actifs pour un classement fiable

**Classement 2 : artistes les plus populaires (moyenne de popularite, minimum 8 morceaux)**

In [29]:
SEUIL_MIN_MORCEAUX = 8

stats_artistes = df_artists.groupby('artists')['popularity'].agg(['mean', 'count'])
top_popularite = stats_artistes[stats_artistes['count'] >= SEUIL_MIN_MORCEAUX].sort_values('mean', ascending=False).head(15)

fig = px.bar(top_popularite, x='mean', y=top_popularite.index, orientation='h', text=top_popularite['mean'].round(1),
             hover_data={'count': True},
             labels={'mean': 'popularite moyenne', 'artists': '', 'count': 'nombre de morceaux'},
             title=f'Classement 2 : les 15 artistes les plus populaires (minimum {SEUIL_MIN_MORCEAUX} morceaux)')
fig.update_yaxes(autorange='reversed')   # artiste le plus populaire en haut
fig.show()

top_popularite


,mean,count
artists,,
Lil Nas X,82.750000,8
Frank Ocean,74.142857,21
Radiohead,73.545455,11
The Neighbourhood,73.481481,27
George Ezra,73.375000,8
System Of A Down,72.750000,12
Stray Kids,72.608696,23
TWICE,72.470588,17
Travis Scott,72.304348,23


**Classement 3 : morceaux les plus populaires**

In [30]:
top_morceaux = df_tracks.sort_values('popularity', ascending=False)[['track_name', 'artists', 'popularity']].head(15)
top_morceaux


,track_name,artists,popularity
20001,Unholy (feat. Kim Petras),Sam Smith;Kim Petras,100
51664,"Quevedo: Bzrp Music Sessions, Vol. 52",Bizarrap;Quevedo,99
20008,I'm Good (Blue),David Guetta;Bebe Rexha,98
67356,La Bachata,Manuel Turizo,98
67358,Me Porto Bonito,Bad Bunny;Chencho Corleone,97
67359,Tití Me Preguntó,Bad Bunny,97
67559,Efecto,Bad Bunny,96
20000,Under The Influence,Chris Brown,96
79000,I Ain't Worried,OneRepublic,96
81052,As It Was,Harry Styles,95


**Interpretation metier** : les classements 1 (presence) et 2 (popularite) ne partagent aucun artiste commun.

- Le **classement 1** reflete le volume de catalogue accumule, pas la demande actuelle : legendes country avec un large back-catalog (George Jones, Hank Williams), compositeurs Bollywood tres prolifiques (Pritam, Arijit Singh), et Mozart (305 morceaux) qui est probablement un artefact du dataset : les mouvements d'une meme symphonie/concerto sont catalogues comme des morceaux distincts, pas 305 chansons independantes.
- Le **classement 2** reflete la demande actuelle du public : artistes contemporains (pop, hip-hop, K-pop, alt-rock) avec un score moyen de streams eleve.

**Pour Camille** : le classement pertinent pour decider quels artistes pousser dans le calendrier editorial est le **classement 2** (popularite), pas le classement 1. Un large catalogue ne signifie pas une demande actuelle du public ; c'est le classement 2 qui reflete l'interet reel des auditeurs aujourd'hui.

**Pourquoi ce classement n'est pas fait sur `df_genre`, contrairement au corrige** : sur `df_genre`, un morceau range dans plusieurs genres compte une fois par genre. Le comptage ci-dessous refait le calcul des deux facons pour quelques artistes.

In [31]:
# Meme calcul que df_artists, mais sur df_genre : un morceau multi-genre y compte plusieurs fois
artistes_genre = df_genre.assign(artists=df_genre['artists'].str.split(';')).explode('artists')
artistes_genre['artists'] = artistes_genre['artists'].str.strip()

exemples = ['Måneskin', 'ROSALÍA', 'LISA', 'George Ezra', 'TWICE']
comparaison_artistes = pd.DataFrame({
    'morceaux comptes sur df_genre': artistes_genre['artists'].value_counts().reindex(exemples),
    'morceaux reels (df_tracks)': df_artists['artists'].value_counts().reindex(exemples),
})
comparaison_artistes


,morceaux comptes sur df_genre,morceaux reels (df_tracks)
artists,,
Måneskin,12,5
ROSALÍA,11,5
LISA,10,4
George Ezra,23,8
TWICE,17,17


**Consequence sur le classement 2** : le meme classement par popularite (seuil de 8 morceaux), recalcule sur `df_genre`. La colonne `morceaux reels` donne le nombre de morceaux de l'artiste dans `df_tracks`.

In [32]:
stats_genre = artistes_genre.groupby('artists')['popularity'].agg(['mean', 'count'])
top_popularite_genre = stats_genre[stats_genre['count'] >= SEUIL_MIN_MORCEAUX].sort_values('mean', ascending=False).head(15)
top_popularite_genre['morceaux reels'] = df_artists['artists'].value_counts().reindex(top_popularite_genre.index)
top_popularite_genre['sous le seuil en reel'] = top_popularite_genre['morceaux reels'] < SEUIL_MIN_MORCEAUX

communs = top_popularite_genre.index.intersection(top_popularite.index)
print(f"Artistes du top 15 sous le seuil de {SEUIL_MIN_MORCEAUX} morceaux en realite : {top_popularite_genre['sous le seuil en reel'].sum()} sur 15")
print(f"Artistes communs aux deux classements : {len(communs)} sur 15")

top_popularite_genre.round(1)


Artistes du top 15 sous le seuil de 8 morceaux en realite : 9 sur 15
Artistes communs aux deux classements : 6 sur 15


,mean,count,morceaux reels,sous le seuil en reel
artists,,,,
Måneskin,83.7,12,5,True
Lil Nas X,83.5,11,8,False
Mitski,78.9,9,6,True
ROSALÍA,77.1,11,5,True
Radiohead,76.6,20,11,False
Regard,76.3,9,4,True
The Neighbourhood,75.6,60,27,False
Two Feet,75.3,9,4,True
Lost Frequencies,74.5,10,5,True


**Conclusion** : sur `df_genre`, le nombre de morceaux d'un artiste est gonfle des que ses titres sont ranges dans plusieurs genres : 12 morceaux pour Maneskin, qui n'en a que 5, et 23 pour George Ezra, qui en a 8. Seul TWICE, dont les titres ne figurent chacun que dans un genre, reste a 17.

Les consequences sur le classement 2 sont importantes : 9 des 15 artistes du top calcule sur `df_genre` ont moins de 8 morceaux reels, dont le premier, Maneskin, avec 5 morceaux. Le seuil, choisi pour ecarter les artistes dont la moyenne repose sur trop peu de morceaux, ne joue plus son role, et les deux classements n'ont que 6 artistes en commun. C'est pourquoi les classements sont calcules sur `df_tracks`, alors que le reste de l'analyse utilise `df_genre` (voir section 3.3).

### 4.7 Comparaison de genres contrastes (radar chart)

Genres retenus : **k-pop** (grand public et energique), **sleep** (niche et calme) et **drum-and-bass** (niche et tres energique). Le choix s'appuie sur le classement de popularite par genre ci-dessous, de maniere a couvrir un genre populaire et deux genres de niche aux profils opposes. `sleep` fait par ailleurs le lien avec l'investigation menee en section 4.4 sur les morceaux a `tempo = 0`.

Calcul sur `df_genre`, avec les sept variables audio deja bornees entre 0 et 1. `loudness` (en dB) et `tempo` (en BPM) sont volontairement ecartees : les placer sur le meme axe radial imposerait une normalisation artificielle qui rendrait la lecture trompeuse.

In [33]:
df_genre.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(15)

track_genre
pop-film             59.280280
k-pop                56.963928
chill                53.704705
sad                  52.379000
grunge               49.582583
indian               49.528529
anime                48.766767
emo                  48.128000
pop                  47.903323
sertanejo            47.866000
progressive-house    46.661662
piano                45.277555
mandopop             45.025000
deep-house           44.788577
brazil               44.668337
Name: popularity, dtype: float64

In [34]:
genres_compares = ['k-pop', 'sleep', 'drum-and-bass']

# Seules les variables deja bornees entre 0 et 1 sont retenues : elles sont directement
# comparables sur un meme axe radial, sans normalisation artificielle.
# loudness (dB) et tempo (BPM) en sont exclus pour cette raison.
radar_cols = ['danceability', 'energy', 'speechiness', 'acousticness',
              'instrumentalness', 'liveness', 'valence']

profil_genres = (df_genre[df_genre['track_genre'].isin(genres_compares)]
                 .groupby('track_genre')[radar_cols].mean()
                 .reindex(genres_compares))
profil_genres.round(3)


,danceability,energy,speechiness,acousticness,instrumentalness,liveness,valence
track_genre,,,,,,,
k-pop,0.648,0.676,0.088,0.292,0.010,0.176,0.557
sleep,0.168,0.342,0.057,0.657,0.724,0.358,0.058
drum-and-bass,0.536,0.876,0.092,0.026,0.372,0.220,0.322


In [35]:
fig = go.Figure()

for genre in genres_compares:
    fig.add_trace(go.Scatterpolar(
        r=profil_genres.loc[genre].tolist(),
        theta=radar_cols,
        name=genre,
        fill='toself',
        opacity=0.50          # transparence, pour que les genres superposes restent lisibles
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(range=[0, 1])),
    title='Profil audio moyen par genre',
    height=600
)
fig.show()


**Interpretation metier** : les trois profils se separent nettement, et c'est `instrumentalness` qui discrimine le mieux.

| Genre | Signature audio |
|---|---|
| **sleep** | instrumentalness 0,72 et acousticness 0,66 dominantes, tout le reste au plancher (valence 0,06, danceability 0,17). Musique acoustique, sans voix, emotionnellement neutre. |
| **drum-and-bass** | energy 0,88 au maximum, acousticness quasi nulle (0,03), instrumentalness notable (0,37). Production electronique intense et largement sans voix. |
| **k-pop** | profil le plus equilibre : danceability 0,65 et valence 0,56 les plus elevees des trois, instrumentalness quasi nulle (0,01). Musique chantee, dansante et positive. |

**Ce que revele l'axe `instrumentalness`** : il separe les trois genres de facon quasi binaire (0,72 / 0,37 / 0,01). La presence ou l'absence de voix est le marqueur le plus discriminant du profil sonore, davantage que l'energie ou le tempo. Or cette variable ne figurait pas dans la demande initiale de Camille : s'en tenir a sa liste aurait fait perdre l'axe le plus informatif du graphique.

**Piste pour Camille** : ces profils suggerent des contextes d'ecoute distincts, une ecoute calme et de fond pour sleep, festive pour le drum-and-bass, active et diurne pour la k-pop. **Cette association a un moment de la journee reste une hypothese** : le dataset ne contient aucune donnee d'horaire d'ecoute permettant de la verifier. A presenter comme piste de reflexion, pas comme un resultat.

### 4.8 Outliers IQR sur loudness

In [36]:
Q1 = df_genre['loudness'].quantile(0.25)
Q3 = df_genre['loudness'].quantile(0.75)
IQR = Q3 - Q1
seuil_bas = Q1 - 1.5 * IQR
seuil_haut = Q3 + 1.5 * IQR

outliers_loudness = df_genre[(df_genre['loudness'] < seuil_bas) | (df_genre['loudness'] > seuil_haut)]

print(f"Seuil bas : {seuil_bas:.2f} dB, seuil haut : {seuil_haut:.2f} dB")
print(f"{len(outliers_loudness)} morceaux hors bornes ({len(outliers_loudness)/len(df_genre)*100:.1f}%)")
outliers_loudness['loudness'].describe()


Seuil bas : -17.49 dB, seuil haut : 2.49 dB
6115 morceaux hors bornes (5.4%)


count    6115.000000
mean      -23.326995
std         5.085938
min       -49.531000
25%       -26.133500
50%       -21.921000
75%       -19.371000
max         4.532000
Name: loudness, dtype: float64

In [37]:
# Distribution de loudness avec les deux seuils IQR
fig = px.histogram(df_genre, x='loudness', nbins=100, title='Distribution de loudness et seuils IQR')
fig.add_vline(x=seuil_bas, line_dash='dash', line_color='red', annotation_text=f'seuil bas {seuil_bas:.1f} dB')
fig.add_vline(x=seuil_haut, line_dash='dash', line_color='red', annotation_text=f'seuil haut {seuil_haut:.1f} dB')
fig.show()

# Genres qui concentrent le plus d'outliers
genres_outliers = outliers_loudness['track_genre'].value_counts().head(10)
fig = px.bar(x=genres_outliers.values, y=genres_outliers.index, orientation='h', text=genres_outliers.values,
             labels={'x': "nombre d'outliers", 'y': ''},
             title=f'Les 10 genres qui concentrent le plus d\'outliers de loudness ({len(outliers_loudness)} au total)')
fig.update_yaxes(autorange='reversed')
fig.show()

print(f"Outliers au-dessus du seuil haut : {(outliers_loudness['loudness'] > seuil_haut).sum()}")
print(f"Part des outliers dans ces 10 genres : {genres_outliers.sum() / len(outliers_loudness) * 100:.1f} %")


Outliers au-dessus du seuil haut : 3
Part des outliers dans ces 10 genres : 70.8 %


**Conclusion** : la methode IQR sur `loudness` (seuils -17,49 dB et 2,49 dB) detecte **6115 outliers**, soit 5,4% des lignes, presque tous du cote bas (morceaux tres silencieux) : seuls 3 depassent le seuil haut. Les 10 genres les plus touches regroupent 70,8 % des outliers, et ce sont des genres calmes, acoustiques ou orchestraux (sleep, classical, new-age, ambient, piano, opera, guitar), ou un mastering discret est normal et recherche, a l'oppose de la pop, du rock et de l'electro compresses au maximum (la 'loudness war').

**Valeurs reelles et legitimes, donnees conservees telles quelles**, aucune suppression. C'est le troisieme cas d'outliers investigue dans ce notebook, apres `tempo = 0` et les durees extremes, et les trois aboutissent a la meme conclusion : un outlier statistique n'est pas une erreur de donnees.

## 5. Tests statistiques (Etape 3)

### 5.1 Matrice de correlation des variables audio

Calculee sur `df_genre`, conformement a la regle de la section 3.3.

In [38]:
corr_cols = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
             'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_sec', 'popularity']
corr_matrix = df_genre[corr_cols].corr()

fig = px.imshow(corr_matrix, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                title='Matrice de correlation des variables audio et de la popularite',
                height=700)
fig.show()


### 5.2 Test de normalite global (Shapiro-Wilk)

Ce test porte sur la distribution **globale** des variables, ce qui est l'hypothese pertinente pour la correlation de la section 5.3 (Q1), calculee sur l'ensemble du catalogue.

Pour la comparaison entre genres (Q2, section 5.4), l'hypothese a verifier est differente : la normalite doit etre testee **a l'interieur de chaque groupe compare**. Ce second test est donc fait separement en 5.4.

Note technique : Shapiro-Wilk devient instable au-dela d'environ 5000 observations, on travaille donc sur un echantillon aleatoire.

In [39]:
from scipy.stats import shapiro

sample = df_genre.sample(5000, random_state=42)

for col in ['energy', 'loudness', 'popularity']:
    stat, p = shapiro(sample[col])
    normal = 'normale' if p > 0.05 else 'PAS normale'
    print(f"{col:12s} -> statistique={stat:.4f}, p-value={p:.2e} => distribution {normal} (seuil 0.05)")


energy       -> statistique=0.9501, p-value=3.14e-38 => distribution PAS normale (seuil 0.05)
loudness     -> statistique=0.8410, p-value=2.48e-57 => distribution PAS normale (seuil 0.05)
popularity   -> statistique=0.9567, p-value=3.74e-36 => distribution PAS normale (seuil 0.05)


In [40]:
# Graphiques de normalite : histogramme avec courbe normale theorique + QQ-plot
from scipy.stats import norm, probplot
import numpy as np

for col in ['energy', 'loudness', 'popularity']:
    valeurs = sample[col]

    # Histogramme avec la courbe normale theorique superposee
    fig = go.Figure()
    fig.add_trace(go.Histogram(x=valeurs, histnorm='probability density', name='Donnees observees', opacity=0.7))
    x_theorique = np.linspace(valeurs.min(), valeurs.max(), 200)
    y_theorique = norm.pdf(x_theorique, valeurs.mean(), valeurs.std())
    fig.add_trace(go.Scatter(x=x_theorique, y=y_theorique, mode='lines', name='Loi normale theorique',
                             line=dict(color='red', width=2)))
    fig.update_layout(title=f'{col} : distribution observee vs loi normale', bargap=0.05)
    fig.show()

    # QQ-plot : si les points suivent la droite, la distribution est normale
    (quantiles_theoriques, quantiles_observes), _ = probplot(valeurs, dist='norm')
    fig_qq = go.Figure()
    fig_qq.add_trace(go.Scatter(x=quantiles_theoriques, y=quantiles_observes, mode='markers',
                                name='Quantiles observes', marker=dict(size=3, opacity=0.5)))
    fig_qq.add_trace(go.Scatter(x=quantiles_theoriques, y=quantiles_theoriques * valeurs.std() + valeurs.mean(),
                                mode='lines', name='Droite de normalite', line=dict(color='red')))
    fig_qq.update_layout(title=f'QQ-plot : {col}', xaxis_title='Quantiles theoriques', yaxis_title='Quantiles observes')
    fig_qq.show()


**Lecture des graphiques** : sur les histogrammes, la courbe rouge represente la loi normale theorique qu'auraient les donnees si elles etaient normalement distribuees. Sur les QQ-plots, des points alignes sur la droite rouge indiqueraient une distribution normale.

Les trois variables s'ecartent nettement de la droite, en particulier aux extremites : `loudness` decroche fortement dans la queue gauche (morceaux tres silencieux des genres calmes, cf. section 4.8) et `popularity` presente un decrochage caracteristique du pic de morceaux a 0.

**Consequence methodologique** : les tests non parametriques (Spearman, Kruskal-Wallis) sont a privilegier sur leurs equivalents parametriques (Pearson, ANOVA) pour les sections suivantes.

### 5.3 Q1 - Les caracteristiques audio sont-elles liees entre elles ? (energy vs loudness)

In [41]:
from scipy.stats import pearsonr, spearmanr

r_pearson, p_pearson = pearsonr(df_genre['energy'], df_genre['loudness'])
r_spearman, p_spearman = spearmanr(df_genre['energy'], df_genre['loudness'])

print(f"Pearson  : r={r_pearson:.3f}, p={p_pearson:.2e}")
print(f"Spearman : r={r_spearman:.3f}, p={p_spearman:.2e}")

fig = px.scatter(df_genre.sample(5000, random_state=42), x='energy', y='loudness', trendline='ols',
                  title='Energy vs Loudness (echantillon 5000 points pour lisibilite)', opacity=0.4)
fig.show()

Pearson  : r=0.761, p=0.00e+00
Spearman : r=0.749, p=0.00e+00


**H0** : il n'y a pas de relation lineaire ou monotone entre energy et loudness.

**Test retenu** : Shapiro-Wilk (section 5.2) rejette la normalite d'`energy` et de `loudness`, Spearman est donc le test techniquement justifie. Cela dit, Pearson (r = 0,761) et Spearman (r = 0,749) donnent des resultats quasi identiques, ce qui indique que la relation est bien lineaire et pas seulement monotone.

**Conclusion metier** : H0 est rejetee (p < 0,001 dans les deux cas). La taille d'effet, avec un r autour de 0,75, est **forte** et c'est la l'information utile : avec plus de 113 000 lignes, la significativite etait acquise d'avance, seul le coefficient dit quelque chose de l'ampleur du lien.

Plus un morceau est energique, plus il est masterise fort. C'est un choix de production coherent, pas un artefact de l'echantillon. **Pour Camille** : les titres dynamiques sont systematiquement produits a un niveau sonore plus eleve, une caracteristique a garder en tete quand on definit le profil sonore attendu d'un genre.

**Deuxieme relation : acousticness et energy**

La matrice de correlation (section 5.1) designe ce couple comme la relation negative la plus marquee du catalogue. Meme demarche que pour energy et loudness : verification de la normalite, puis Pearson et Spearman.

In [42]:
# Normalite d'acousticness, sur le meme echantillon que la section 5.2
stat_ac, p_ac = shapiro(sample['acousticness'])
print(f"Shapiro-Wilk acousticness : statistique={stat_ac:.4f}, p-value={p_ac:.2e}")

r_pearson_ac, p_pearson_ac = pearsonr(df_genre['acousticness'], df_genre['energy'])
r_spearman_ac, p_spearman_ac = spearmanr(df_genre['acousticness'], df_genre['energy'])

print(f"Pearson  : r={r_pearson_ac:.3f}, p={p_pearson_ac:.2e}")
print(f"Spearman : r={r_spearman_ac:.3f}, p={p_spearman_ac:.2e}")

# Lecture concrete : energie moyenne des morceaux tres acoustiques contre tres produits
tres_acoustiques = df_genre[df_genre['acousticness'] > 0.9]
tres_produits = df_genre[df_genre['acousticness'] < 0.1]
print(f"\nMorceaux tres acoustiques (> 0.9) : {len(tres_acoustiques) / len(df_genre) * 100:.1f} %, energie moyenne {tres_acoustiques['energy'].mean():.2f}")
print(f"Morceaux tres produits (< 0.1)    : {len(tres_produits) / len(df_genre) * 100:.1f} %, energie moyenne {tres_produits['energy'].mean():.2f}")

fig = px.scatter(df_genre.sample(5000, random_state=42), x='acousticness', y='energy', trendline='ols',
                 title='Acousticness vs Energy (echantillon 5000 points pour lisibilite)', opacity=0.4)
fig.show()


Shapiro-Wilk acousticness : statistique=0.8426, p-value=3.82e-57
Pearson  : r=-0.733, p=0.00e+00
Spearman : r=-0.707, p=0.00e+00

Morceaux tres acoustiques (> 0.9) : 7.8 %, energie moyenne 0.20
Morceaux tres produits (< 0.1)    : 42.4 %, energie moyenne 0.80


**H0** : il n'y a pas de relation lineaire ou monotone entre acousticness et energy.

**Test retenu** : Shapiro-Wilk rejette la normalite d'`acousticness` (p = 3,8e-57), comme pour `energy` en section 5.2 : Spearman est donc le test justifie. Pearson (r = -0,733) et Spearman (r = -0,707) restent tres proches.

**Conclusion metier** : H0 est rejetee (p < 0,001). La relation est **negative et forte** : plus un morceau est acoustique, moins il est energique. Les morceaux tres acoustiques (7,8 % du catalogue) ont une energie moyenne de 0,20, contre 0,80 pour les morceaux tres produits (42,4 % du catalogue). Le nuage de points montre d'ailleurs une forte concentration des morceaux pres d'une acousticness nulle : le catalogue est majoritairement fait de musique produite.

**Reponse a la question du comite** : oui, les caracteristiques audio sont liees entre elles. Energy, loudness et acousticness decrivent en grande partie une meme opposition, entre musique produite, forte et energique d'un cote, et musique acoustique, calme et peu amplifiee de l'autre. **Pour Camille** : pour profiler un morceau, ces trois indicateurs se recoupent largement ; danceability, valence ou speechiness apportent davantage d'information nouvelle.

**La relation energy / loudness est-elle la meme dans tous les genres ?**

Le r = 0,761 calcule plus haut porte sur tout le catalogue. On le recalcule a l'interieur des trois genres deja compares au radar chart (section 4.7).

In [43]:
df_trois_genres = df_genre[df_genre['track_genre'].isin(genres_compares)]

for genre in genres_compares:
    sous_ensemble = df_trois_genres[df_trois_genres['track_genre'] == genre]
    r, p = spearmanr(sous_ensemble['energy'], sous_ensemble['loudness'])
    print(f"{genre:15s} Spearman r={r:.3f}, p={p:.2e}")

fig = px.scatter(df_trois_genres, x='energy', y='loudness', color='track_genre', trendline='ols', opacity=0.4,
                 labels={'track_genre': 'genre'},
                 title='Energy vs Loudness, une droite par genre')
fig.show()


k-pop           Spearman r=0.699, p=2.93e-147
sleep           Spearman r=0.420, p=5.73e-44
drum-and-bass   Spearman r=0.541, p=1.01e-76


**Interpretation** : la relation reste positive et significative dans chacun des trois genres, mais elle est plus faible qu'a l'echelle du catalogue (Spearman 0,749) : 0,699 pour la k-pop, 0,541 pour le drum-and-bass et 0,420 pour le sleep.

Une partie de la correlation globale vient donc des **differences entre genres** : les genres calmes sont a la fois peu energiques et peu forts, les genres electroniques a la fois energiques et forts, ce qui etire le nuage de points sur toute la diagonale. A l'interieur d'un genre, l'ecart est plus resserre, et le volume depend aussi d'autres facteurs comme le mastering.

**Pour Camille** : energy et loudness vont ensemble dans tous les genres, mais on ne peut pas deduire le volume d'un morceau de sleep de son energie aussi surement que pour un morceau de k-pop.

### 5.4 Q2 - La popularite differe-t-elle vraiment selon le genre ? (df_genre, genre = variable etudiee)

**Verification prealable : normalite a l'interieur de chaque genre**

L'ANOVA suppose que la variable est normalement distribuee **dans chaque groupe compare**, et non que sa distribution globale soit normale. C'est cette hypothese qu'il faut tester avant de choisir entre ANOVA et Kruskal-Wallis.

In [44]:
# Shapiro-Wilk applique genre par genre (environ 1000 morceaux par genre, taille adaptee au test)
resultats_normalite = []
for genre, groupe in df_genre.groupby('track_genre'):
    stat_sw, p_sw = shapiro(groupe['popularity'])
    resultats_normalite.append({'genre': genre, 'n': len(groupe),
                                'shapiro_stat': stat_sw, 'p_value': p_sw,
                                'normal': p_sw > 0.05})

normalite_df = pd.DataFrame(resultats_normalite)

nb_normaux = normalite_df['normal'].sum()
print(f"Genres testes : {len(normalite_df)}")
print(f"Genres dont la popularite suit une loi normale : {nb_normaux}")
print(f"Genres non normaux : {len(normalite_df) - nb_normaux}")

normalite_df.sort_values('p_value', ascending=False).head(10)


Genres testes : 114
Genres dont la popularite suit une loi normale : 0
Genres non normaux : 114


,genre,n,shapiro_stat,p_value,normal
5,anime,999,0.951896,1.481980e-17,False
107,synth-pop,1000,0.948955,4.072034e-18,False
69,malay,999,0.948557,3.521830e-18,False
105,study,998,0.946189,1.354734e-18,False
12,cantopop,999,0.943596,4.695395e-19,False
113,world-music,999,0.937268,4.294310e-20,False
109,techno,1000,0.937163,4.035526e-20,False
43,groove,998,0.936976,3.957395e-20,False
14,children,998,0.934769,1.797258e-20,False
0,acoustic,1000,0.934503,1.557771e-20,False


In [45]:
from scipy.stats import kruskal, f_oneway

# Regle de decision explicite : le critere de choix du test est dans le code, pas seulement en commentaire
tous_normaux = normalite_df['normal'].all()
groupes = [g['popularity'].values for _, g in df_genre.groupby('track_genre')]

if tous_normaux:
    stat, p = f_oneway(*groupes)
    test_retenu = 'ANOVA'
else:
    stat, p = kruskal(*groupes)
    test_retenu = 'Kruskal-Wallis'

print(f"Normalite verifiee dans tous les genres : {tous_normaux}")
print(f"Test retenu : {test_retenu}")
print(f"Statistique = {stat:.2f}, p-value = {p:.2e}")
print('Difference significative entre genres' if p < 0.05 else 'Pas de difference significative')


Normalite verifiee dans tous les genres : False
Test retenu : Kruskal-Wallis
Statistique = 30955.26, p-value = 0.00e+00
Difference significative entre genres


In [46]:
# Test de controle : ANOVA calculee malgre la non-normalite, pour verifier que la conclusion ne depend pas du test choisi
stat_kruskal, p_kruskal = kruskal(*groupes)
stat_anova, p_anova = f_oneway(*groupes)

print(f"Kruskal-Wallis : statistique={stat_kruskal:.2f}, p-value={p_kruskal:.2e}")
print(f"ANOVA          : statistique={stat_anova:.2f}, p-value={p_anova:.2e}")
print('Les deux tests concordent' if (p_kruskal < 0.05) == (p_anova < 0.05) else 'Les deux tests divergent')


Kruskal-Wallis : statistique=30955.26, p-value=0.00e+00
ANOVA          : statistique=338.26, p-value=0.00e+00
Les deux tests concordent


In [47]:
# Boxplot sur un sous-ensemble de genres pour lisibilite (top 10 par nombre de morceaux)
top_genres = df_genre['track_genre'].value_counts().head(10).index
fig = px.box(df_genre[df_genre['track_genre'].isin(top_genres)], x='track_genre', y='popularity',
             title='Popularite par genre (10 genres les plus representes)')
fig.show()


In [48]:
# Taille d'effet pour Kruskal-Wallis : epsilon carre
# Interpretation : part de la variance de popularite expliquee par le genre (0 a 1)
n_total = len(df_genre)
k_groupes = df_genre['track_genre'].nunique()
epsilon_carre = (stat - k_groupes + 1) / (n_total - k_groupes)

print(f"Nombre d'observations : {n_total}, nombre de genres : {k_groupes}")
print(f"Epsilon carre = {epsilon_carre:.4f}")
print(f"=> le genre explique environ {epsilon_carre*100:.1f}% de la variance de la popularite")


Nombre d'observations : 113549, nombre de genres : 114
Epsilon carre = 0.2719
=> le genre explique environ 27.2% de la variance de la popularite


**Taille d'effet** : epsilon carre = 0.272, soit **27,2% de la variance de popularite expliquee par le genre**.

Reperes usuels pour l'epsilon carre : 0.01 = effet faible, 0.06 = moyen, 0.14 = fort. A 0.27, l'effet est donc **nettement fort**.

C'est l'information que la p-value ne donne pas : avec 113 549 observations, la significativite etait acquise d'avance. La taille d'effet confirme que le genre est un facteur reellement structurant de la popularite, et non une difference marginale rendue significative par le volume de donnees.

**Test post-hoc (Dunn)** : Kruskal-Wallis indique qu'au moins un genre differe des autres, mais pas lesquels. Le test de Dunn compare les genres deux a deux, avec correction de Bonferroni pour tenir compte du nombre eleve de comparaisons.

Applique aux 10 genres les plus representes pour rester lisible (comparer les 114 genres donnerait plus de 6400 paires).

In [49]:
import scikit_posthocs as sp

sous_ensemble = df_genre[df_genre['track_genre'].isin(top_genres)]
resultats_dunn = sp.posthoc_dunn(sous_ensemble, val_col='popularity', group_col='track_genre',
                                  p_adjust='bonferroni')

fig = px.imshow(resultats_dunn.round(4), text_auto='.3f', color_continuous_scale='RdYlGn',
                zmin=0, zmax=0.05,
                title='Test de Dunn : p-values par paire de genres (vert = difference significative)')
fig.show()

# Nombre de paires significativement differentes
import numpy as np
masque = np.triu(np.ones(resultats_dunn.shape), k=1).astype(bool)
paires = resultats_dunn.where(masque).stack()
print(f"Paires de genres comparees : {len(paires)}")
print(f"Paires significativement differentes (p < 0.05) : {(paires < 0.05).sum()}")


Paires de genres comparees : 45
Paires significativement differentes (p < 0.05) : 38


**Resultat du test de Dunn** : sur les 45 paires comparees parmi les 10 genres les plus representes, **38 presentent une difference significative** (84%), apres correction de Bonferroni.

Le test de Kruskal-Wallis disait seulement qu'au moins un genre se distinguait ; Dunn montre que la difference est generalisee, et non portee par un ou deux genres atypiques. Les 7 paires non significatives correspondent a des genres au profil de popularite comparable.

**Pour Camille** : les genres ne se repartissent pas en deux blocs (populaires contre confidentiels), chacun occupe une position distincte. Une strategie editoriale differenciee genre par genre se justifie donc, plutot qu'un traitement par grandes categories.

**H0** : la popularite moyenne est la meme dans tous les genres.

Kruskal-Wallis choisi plutot qu'ANOVA car Shapiro-Wilk rejette la normalite (section 5.2).

**Conclusion metier** : H0 rejetee dans les deux tests (Kruskal-Wallis stat=30955, ANOVA stat=338, p < 0.001). Les deux methodes s'accordent malgre la non-normalite : avec un ecart aussi net, le resultat est robuste au choix du test. Le boxplot montre des profils tres differents selon le genre : reggaeton et disco ont une mediane basse mais une tres grande dispersion (quelques hits tres populaires, beaucoup de morceaux confidentiels), tandis que mpb est tres resserre autour d'une popularite moderee (audience de niche mais constante). **Pour Camille** : la popularite depend reellement du genre, mais pas de la meme facon. Certains genres sont 'a hits' (grande variance, quelques titres portent tout le genre), d'autres sont 'stables' (audience fidele, popularite homogene). Ca oriente differemment la strategie de mise en avant selon le genre.

### 5.5 Q3 - Le contenu explicite depend-il du genre ? (df_genre, khi2)

In [50]:
from scipy.stats import chi2_contingency

table_contingence = pd.crosstab(df_genre['track_genre'], df_genre['explicit'])
chi2, p, ddl, attendu = chi2_contingency(table_contingence)

print(f"Khi2 = {chi2:.2f}, ddl = {ddl}, p-value = {p:.2e}")
table_contingence.head(10)


Khi2 = 18362.08, ddl = 113, p-value = 0.00e+00


explicit,False,True
track_genre,,
acoustic,948,52
afrobeat,981,18
alt-rock,943,56
alternative,835,164
ambient,994,5
anime,944,55
black-metal,866,131
bluegrass,993,5
blues,984,14


In [51]:
# Taille d'effet pour le khi2 : V de Cramer
# Interpretation : force de l'association entre les deux variables qualitatives (0 = aucune, 1 = totale)
n = table_contingence.sum().sum()
dim_min = min(table_contingence.shape) - 1
v_cramer = (chi2 / (n * dim_min)) ** 0.5

print(f"V de Cramer = {v_cramer:.4f}")
print("Reperes usuels : 0.1 = faible, 0.3 = moyen, 0.5 = fort")


V de Cramer = 0.4021
Reperes usuels : 0.1 = faible, 0.3 = moyen, 0.5 = fort


**Taille d'effet** : V de Cramer = 0.402.

Sur l'echelle usuelle (0.1 faible, 0.3 moyen, 0.5 fort), l'association entre genre et contenu explicite se situe **entre moyenne et forte**.

Le khi² de 18 362 impressionne mais ne dit rien de l'ampleur : il augmente mecaniquement avec la taille de l'echantillon. Le V de Cramer, lui, est normalise entre 0 et 1 et reste interpretable quel que soit le volume. A 0.40, le genre est un predicteur solide du caractere explicite d'un morceau, sans etre deterministe.

In [52]:
# Graphique bivarie pour le khi2 : taux de contenu explicite par genre
taux_explicite = (df_genre.groupby('track_genre')['explicit'].mean() * 100).sort_values(ascending=False)

# Les 15 genres les plus explicites et les 15 les moins explicites
extremes = pd.concat([taux_explicite.head(15), taux_explicite.tail(15)])

fig = px.bar(x=extremes.index, y=extremes.values,
             labels={'x': 'genre', 'y': '% de morceaux explicites'},
             title='Taux de contenu explicite par genre (15 genres les plus et les moins concernes)')
fig.add_hline(y=df_genre['explicit'].mean() * 100, line_dash='dash', line_color='red',
              annotation_text='moyenne du catalogue')
fig.update_layout(xaxis_tickangle=-45)
fig.show()


**H0** : le caractere explicite est independant du genre.

**Conclusion metier** : H0 rejetee (Khi2=18362, ddl=113, p < 0.001) : le contenu explicite depend fortement du genre. Le tableau de contingence le confirme visuellement : quasi aucun morceau explicite en ambient (5/999), bluegrass (5/998) ou afrobeat (18/999), contre une proportion beaucoup plus elevee en alternative (164/999, ~16%) ou black-metal (131/997, ~13%). **Pour Camille** : le taux de contenu explicite varie enormement d'un genre a l'autre, ce qui est utile pour calibrer la moderation de contenu et les recommandations selon le public cible (playlists familiales vs genres plus radicaux).

## 6. Analyses complementaires (bonus)

*Analyses menees sur `df_tracks` (dedupliquee) : elles portent sur les caracteristiques du morceau, pas sur le genre.*

### 6.1 Featuring vs solo

In [53]:
from scipy.stats import mannwhitneyu

df_tracks['has_feat'] = df_tracks['artists'].str.contains(';')

print(df_tracks.groupby('has_feat')['popularity'].agg(['mean', 'count']))

stat, p = mannwhitneyu(df_tracks[df_tracks['has_feat']]['popularity'], df_tracks[~df_tracks['has_feat']]['popularity'])
print(f"\nMann-Whitney U : statistique={stat:.0f}, p-value={p:.2e}")


               mean  count
has_feat                  
False     32.856911  67154
True      34.215355  22586

Mann-Whitney U : statistique=792595206, p-value=2.72e-24


### 6.2 Explicite vs popularite (tous genres confondus)

In [54]:
print(df_tracks.groupby('explicit')['popularity'].agg(['mean', 'count']))

stat, p = mannwhitneyu(df_tracks[df_tracks['explicit']]['popularity'], df_tracks[~df_tracks['explicit']]['popularity'])
print(f"\nMann-Whitney U : statistique={stat:.0f}, p-value={p:.2e}")


               mean  count
explicit                  
False     32.852577  82036
True      36.885644   7704

Mann-Whitney U : statistique=348321122, p-value=4.72e-50


### 6.3 Tonalite musicale (key / mode)

In [55]:
# mode : 1 = majeur, 0 = mineur
print(df_tracks.groupby('mode')['popularity'].agg(['mean', 'count']))
print()
print(df_tracks.groupby('key')['popularity'].agg(['mean', 'count']).sort_values('mean', ascending=False))


           mean  count
mode                  
0     33.641476  32578
1     32.946520  57162

          mean  count
key                  
4    34.060564   7133
11   33.881891   7129
8    33.791023   5570
2    33.684143   9327
6    33.483141   6139
3    33.248465   2769
5    32.967843   7308
9    32.874861   8998
1    32.860891   8576
10   32.843437   5889
0    32.721889  10352
7    32.611659  10550


In [56]:
# Popularite moyenne pour chaque combinaison tonalite x mode
noms_tonalites = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']

pivot_tonalite = df_tracks.pivot_table(index='mode', columns='key', values='popularity', aggfunc='mean')
pivot_tonalite.index = ['mineur', 'majeur']   # mode 0 = mineur, 1 = majeur
pivot_tonalite.columns = noms_tonalites

fig = px.imshow(pivot_tonalite.round(1), text_auto=True, color_continuous_scale='Blues', aspect='auto',
                labels={'x': 'tonalite', 'y': 'mode', 'color': 'popularite moyenne'},
                title='Popularite moyenne par tonalite et par mode')
fig.show()

print(f"Ecart entre la case la plus populaire et la moins populaire : {pivot_tonalite.max().max() - pivot_tonalite.min().min():.1f} points sur 100")


Ecart entre la case la plus populaire et la moins populaire : 3.2 points sur 100


**Lecture de la heatmap** : la popularite moyenne varie de 31,9 (do diese majeur) a 35,1 (mi majeur et sol diese mineur), soit 3,2 points sur 100 seulement. Le mode mineur depasse le majeur dans 8 tonalites sur 12, mais sans regularite exploitable : aucune combinaison tonalite x mode ne se detache nettement.

Le croisement confirme donc la conclusion tiree de chaque variable prise separement : la tonalite n'a pas d'influence pratique sur la popularite.

### 6.4 One-hit wonders vs artistes prolifiques

In [57]:
compte_artistes = df_artists.groupby('artists')['popularity'].agg(['mean', 'count'])

one_hit = compte_artistes[compte_artistes['count'] == 1]
prolifiques = compte_artistes[compte_artistes['count'] >= 8]

print(f"One-hit wonders (1 morceau) : {len(one_hit)} artistes, popularite moyenne = {one_hit['mean'].mean():.1f}")
print(f"Artistes prolifiques (>=8 morceaux) : {len(prolifiques)} artistes, popularite moyenne des moyennes = {prolifiques['mean'].mean():.1f}")

stat, p = mannwhitneyu(one_hit['mean'], prolifiques['mean'])
print(f"\nMann-Whitney U : statistique={stat:.0f}, p-value={p:.2e}")


One-hit wonders (1 morceau) : 15007 artistes, popularite moyenne = 36.2
Artistes prolifiques (>=8 morceaux) : 3593 artistes, popularite moyenne des moyennes = 32.8

Mann-Whitney U : statistique=29779244, p-value=1.80e-22


**Interpretations** :

- **6.1 Featuring vs solo** : featuring 34.2 vs solo 32.9, ecart significatif (Mann-Whitney p=2.7e-24) mais modeste (~1.3 points sur 100). Un featuring booste legerement la popularite, sans effet spectaculaire.

- **6.2 Explicite vs popularite** : explicite 36.9 vs non-explicite 32.9, ecart plus marque et significatif (p=4.7e-50). Le contenu explicite est associe a une popularite plus elevee en moyenne, tous genres confondus. A nuancer : correlation, pas forcement causalite (les genres a forte proportion d'explicite comme le hip-hop sont aussi des genres populaires en soi).

- **6.3 Tonalite (key/mode)** : quasi aucune difference. Mode mineur (33.6) legerement au-dessus du majeur (32.9), et les 12 tonalites varient entre 32.6 et 34.1 seulement, un ecart negligeable compare aux autres facteurs etudies. Confirme l'intuition : la tonalite n'a pas d'influence pratique sur la popularite.

- **6.4 One-hit wonders vs prolifiques** : resultat le plus surprenant : les one-hit wonders (36.2) sont en moyenne PLUS populaires que les artistes prolifiques (32.8), difference significative (p=1.8e-22). **Pour Camille** : un artiste avec un seul titre presente sur la plateforme n'est pas forcement moins interessant, certains one-hit wonders touchent tres fort un public precis. Ca nuance le classement 2 de la section 4.6 : le seuil de 8 morceaux exclut peut-etre des pepites a fort potentiel, pas seulement du bruit statistique.

### 6.5 Les variables hors demande de Camille : fallait-il les exploiter ?

`speechiness`, `instrumentalness`, `liveness` et `time_signature` sont presentes dans le dataset mais absentes du mail de Camille. Les trois premieres ont ete integrees aux statistiques descriptives (4.1) et a la matrice de correlation (5.1). Cette section verifie si leur absence de la demande initiale etait justifiee, en comparant leur lien avec la popularite a celui des variables demandees.

*Analyses menees sur `df_tracks`, comme le reste de la section 6.*

In [58]:
cols_inexploitees = ['speechiness', 'instrumentalness', 'liveness', 'time_signature']

df_tracks[cols_inexploitees].describe().round(3)


,speechiness,instrumentalness,liveness,time_signature
count,89740.000,89740.000,89740.000,89740.000
mean,0.087,0.173,0.217,3.897
std,0.113,0.324,0.195,0.453
min,0.000,0.000,0.000,0.000
25%,0.036,0.000,0.098,4.000
50%,0.049,0.000,0.132,4.000
75%,0.086,0.098,0.279,4.000
max,0.965,1.000,1.000,5.000


In [59]:
# Lien de chaque variable audio avec la popularite, variables demandees et hors demande confondues
toutes_variables = audio_cols + ['time_signature']
correlations = df_tracks[toutes_variables + ['popularity']].corr()['popularity'].drop('popularity')
correlations = correlations.reindex(correlations.abs().sort_values().index)   # de la plus faible a la plus forte

tableau = pd.DataFrame({'variable': correlations.index,
                        'correlation': correlations.values,
                        'origine': ['hors demande' if v in cols_inexploitees else 'demande de Camille'
                                    for v in correlations.index]})

fig = px.bar(tableau, x='correlation', y='variable', color='origine', orientation='h',
             category_orders={'variable': list(tableau['variable'])},
             labels={'correlation': 'correlation avec la popularite (r)', 'variable': '', 'origine': ''},
             title='Lien de chaque variable audio avec la popularite')
fig.show()

print(correlations.round(3).iloc[::-1].to_string())


instrumentalness   -0.127
loudness            0.072
danceability        0.064
speechiness        -0.047
acousticness       -0.039
time_signature      0.037
duration_sec       -0.023
liveness           -0.014
energy              0.014
valence            -0.012
tempo               0.007


In [60]:
for col in cols_inexploitees:
    fig = px.histogram(df_tracks, x=col, nbins=50, title=f'Distribution de {col}')
    fig.show()


**Interpretation** : le graphique compare les 11 variables audio, et le resultat contredit l'hypothese de depart.

- **La variable la plus liee a la popularite est hors demande** : `instrumentalness` (r = -0,127) devance toutes les variables du mail de Camille, la premiere etant `loudness` (r = 0,072). Les morceaux instrumentaux sont un peu moins populaires, ce qui est coherent avec la domination du format chante dans le streaming.
- Les trois autres variables hors demande (`speechiness` -0,047, `time_signature` 0,037, `liveness` -0,014) se situent au meme niveau que les variables demandees. `time_signature` est de plus quasi constante (mediane et 75e percentile a 4 temps) : elle ne distingue presque aucun morceau.
- **Toutes les correlations restent faibles** (moins de 0,13 en valeur absolue) : aucune caracteristique audio, demandee ou non, n'explique a elle seule la popularite. A titre de comparaison, le genre en explique environ 27 % (epsilon carre, section 5.4).

**Conclusion** : les donnees ne justifient pas l'exclusion de ces variables, en particulier d'`instrumentalness`, qu'il faudrait integrer a toute analyse future de la popularite. Mais le constat principal est ailleurs. **Pour Camille** : la popularite d'un morceau depend beaucoup plus de son genre (et sans doute de l'artiste et de sa promotion, absents du dataset) que de son profil sonore. Pour recommander des morceaux aux labels, le genre est un meilleur levier que les caracteristiques audio.

## 7. Annexe : explorations ponctuelles

Verifications exploratoires menees en cours d'analyse, conservees pour tracabilite mais non rattachees au raisonnement principal.

In [61]:
df_tracks[df_tracks['artists'].str.contains('Gims', case=False, na=False)][['track_name', 'artists', 'track_genre', 'popularity']]

,track_name,artists,track_genre,popularity
36152,Hola Señorita,GIMS;Maluma,french,62
36470,Bella,GIMS,french,63
36507,Mi Gna (feat. Hayko) - Maître Gims Remix,GIMS;Super Sako;Hayko,french,53
36561,Est-ce que tu m'aimes ? - Pilule bleue,GIMS,french,47
36800,Corazón,GIMS;Lil Wayne;French Montana,french,38
68711,Arhbo [Music from the FIFA World Cup Qatar 202...,Ozuna;GIMS;RedOne;FIFA Sound,latino,63


In [62]:
[g for g in df_tracks['track_genre'].unique() if 'dub' in g.lower() or 'jungle' in g.lower()]

['dub', 'dubstep']